# Phase 9: Train-Test Split

## Objectives
- Load the feature-engineered datasets.
- Separate input features (X) and target labels (y).
- Create stratified 80:20 training and testing sets.
- Verify target distributions and dataset dimensions.
- Save the splits for reproducible model development.

## Research safeguards
- Keep the test set separate from training.
- Preserve class proportions using stratification.
- Do not apply SMOTE before splitting.
- Use the same random seed for reproducibility.

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split

current_dir = Path.cwd()

if (current_dir / "data" / "processed" / "model_ready").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "processed" / "model_ready").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate data/processed/model_ready."
    )

input_dir = project_root / "data" / "processed" / "model_ready"
output_dir = project_root / "data" / "processed" / "splits"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Input directory:", input_dir)
print("Output directory:", output_dir)

Project root: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction
Input directory: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\model_ready
Output directory: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\splits


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "performance"

datasets = {
    "mat": input_dir / "student_mat_features.csv",
    "por": input_dir / "student_por_features.csv",
}

splits = {}

for subject, path in datasets.items():
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    df = pd.read_csv(path)

    if TARGET not in df.columns:
        raise ValueError(f"Missing target column in {path.name}")

    if df[TARGET].isna().any():
        raise ValueError(f"Missing target values in {path.name}")

    X = df.drop(columns=[TARGET])
    y = df[TARGET].astype(int)

    if y.nunique() != 2:
        raise ValueError(
            f"Expected binary target for {path.name}; "
            f"found {sorted(y.unique())}"
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    splits[subject] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }

    print(f"\n{subject.upper()} dataset")
    print("Original shape:", df.shape)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_test:", y_test.shape)

    print("\nOriginal target proportions:")
    print(y.value_counts(normalize=True).sort_index().round(3))

    print("\nTraining target proportions:")
    print(y_train.value_counts(normalize=True).sort_index().round(3))

    print("\nTesting target proportions:")
    print(y_test.value_counts(normalize=True).sort_index().round(3))


MAT dataset
Original shape: (395, 31)
X_train: (316, 30)
X_test: (79, 30)
y_train: (316,)
y_test: (79,)

Original target proportions:
performance
0    0.329
1    0.671
Name: proportion, dtype: float64

Training target proportions:
performance
0    0.329
1    0.671
Name: proportion, dtype: float64

Testing target proportions:
performance
0    0.329
1    0.671
Name: proportion, dtype: float64

POR dataset
Original shape: (649, 31)
X_train: (519, 30)
X_test: (130, 30)
y_train: (519,)
y_test: (130,)

Original target proportions:
performance
0    0.154
1    0.846
Name: proportion, dtype: float64

Training target proportions:
performance
0    0.154
1    0.846
Name: proportion, dtype: float64

Testing target proportions:
performance
0    0.154
1    0.846
Name: proportion, dtype: float64


In [3]:
for subject, split in splits.items():
    for name, data in split.items():
        output_path = output_dir / f"{subject}_{name}.csv"

        if name.startswith("X_"):
            data.to_csv(output_path, index=False)
        else:
            data.rename(TARGET).to_frame().to_csv(
                output_path, index=False
            )

        print("Saved:", output_path.relative_to(project_root))

Saved: data\processed\splits\mat_X_train.csv
Saved: data\processed\splits\mat_X_test.csv
Saved: data\processed\splits\mat_y_train.csv
Saved: data\processed\splits\mat_y_test.csv
Saved: data\processed\splits\por_X_train.csv
Saved: data\processed\splits\por_X_test.csv
Saved: data\processed\splits\por_y_train.csv
Saved: data\processed\splits\por_y_test.csv


In [4]:
for subject in datasets:
    expected_files = [
        f"{subject}_X_train.csv",
        f"{subject}_X_test.csv",
        f"{subject}_y_train.csv",
        f"{subject}_y_test.csv",
    ]

    for filename in expected_files:
        path = output_dir / filename
        assert path.exists(), f"Missing file: {filename}"

        saved = pd.read_csv(path)
        assert len(saved) > 0, f"Empty file: {filename}"

        print(f"{filename}: {saved.shape} — verified")

print("\nTrain-test split completed successfully.")

mat_X_train.csv: (316, 30) — verified
mat_X_test.csv: (79, 30) — verified
mat_y_train.csv: (316, 1) — verified
mat_y_test.csv: (79, 1) — verified
por_X_train.csv: (519, 30) — verified
por_X_test.csv: (130, 30) — verified
por_y_train.csv: (519, 1) — verified
por_y_test.csv: (130, 1) — verified

Train-test split completed successfully.
